In [1]:
from datasets import load_dataset

ds = load_dataset(
    "jacktol/ATC-ASR-Dataset", token=""
)

In [2]:
ds.keys()

dict_keys(['train', 'validation', 'test'])

In [3]:
from paths import DATA_DIR

for key in ds.keys():
    print(f"Key: {key}")
    print(f"Number of samples: {len(ds[key])}")
    print(f"Sample data: {ds[key][0]}")
    print("-" * 40)
    ds[key].to_csv(DATA_DIR / f"atc_asr_dataset_{key}.csv")

Key: train
Number of samples: 6497


RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.6.0+cu124) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent call

In [4]:
import polars as pl

train_df = pl.read_csv("atc_asr_dataset_train.csv")
validation_df = pl.read_csv("atc_asr_dataset_validation.csv")
test_df = pl.read_csv("atc_asr_dataset_test.csv")

In [5]:
train_df.columns

['id', 'audio', 'text']

In [6]:
train_df.shape

(6497, 3)

In [7]:
import ast
import soundfile as sf
import io


def load_audio_to_numpy(audio_str):
    loaded = ast.literal_eval(audio_str)  # parses Python literal syntax
    audio_array, sample_rate = sf.read(io.BytesIO(loaded["bytes"]))
    return audio_array, sample_rate

In [8]:
train_df["audio"][0]

'{\'bytes\': b\'RIFF\\xa4\\xb1\\x00\\x00WAVEfmt \\x10\\x00\\x00\\x00\\x01\\x00\\x01\\x00\\x80>\\x00\\x00\\x00}\\x00\\x00\\x02\\x00\\x10\\x00data\\x80\\xb1\\x00\\x00\\xf8\\xff\\xec\\xff\\xdb\\xff\\xd2\\xff\\xdc\\xff\\xef\\xff\\xfb\\xff\\x00\\x00\\x0c\\x00\\x1c\\x00\\x18\\x00\\xf1\\xff\\xcb\\xff\\xd3\\xff\\x04\\x00)\\x00\\x1b\\x00\\xf2\\xff\\xe4\\xff\\xfc\\xff\\x12\\x00\\x08\\x00\\xed\\xff\\xe8\\xff\\x02\\x00\\x1e\\x00%\\x00\\x1d\\x00\\x1a\\x00\\x1f\\x00"\\x00 \\x00!\\x00(\\x00*\\x00\\x1e\\x00\\r\\x00\\x04\\x00\\x06\\x00\\x0b\\x00\\x11\\x00\\x15\\x00\\x13\\x00\\x03\\x00\\xf4\\xff\\xfb\\xff\\x0f\\x00\\x10\\x00\\xf4\\xff\\xe1\\xff\\xf3\\xff\\x11\\x00\\x08\\x00\\xdc\\xff\\xc3\\xff\\xda\\xff\\xfc\\xff\\xfa\\xff\\xe2\\xff\\xe1\\xff\\xf9\\xff\\x01\\x00\\xee\\xff\\xe5\\xff\\xfd\\xff\\x19\\x00\\x12\\x00\\xf5\\xff\\xed\\xff\\x03\\x00\\x1a\\x00\\x1a\\x00\\x12\\x00\\x10\\x00\\x11\\x00\\n\\x00\\x02\\x00\\x05\\x00\\x11\\x00\\x16\\x00\\x12\\x00\\r\\x00\\x0c\\x00\\x0f\\x00\\x13\\x00\\x15\\x00\\x10\\x00

In [9]:
audio_array, sample_rate = load_audio_to_numpy(train_df["audio"][0])

In [10]:
train_audio = [load_audio_to_numpy(audio_str) for audio_str in train_df["audio"]]
validation_audio = [
    load_audio_to_numpy(audio_str) for audio_str in validation_df["audio"]
]
test_audio = [load_audio_to_numpy(audio_str) for audio_str in test_df["audio"]]

In [11]:
print(
    f"First training audio shape: {train_audio[0][0].shape}, sample rate: {train_audio[0][1]}"
)

First training audio shape: (22720,), sample rate: 16000


In [12]:
train_audio = [
    audio_array for audio_array, sample_rate in train_audio if sample_rate == 16000
]
val_audio = [
    audio_array for audio_array, sample_rate in validation_audio if sample_rate == 16000
]
test_audio = [
    audio_array for audio_array, sample_rate in test_audio if sample_rate == 16000
]

In [13]:
len(train_audio), len(val_audio), len(test_audio)

(6497, 812, 813)

In [14]:
train_df.columns

['id', 'audio', 'text']

In [15]:
train_df["id"][0]

'0023HAQRADETCJDF66RO'

In [16]:
import json
from pathlib import Path

train_path = Path("../data/atc_asr_train")
train_path.mkdir(parents=True, exist_ok=True)
manifest_path = train_path / "manifest.jsonl"
audio_path = train_path / "audio"
audio_path.mkdir(parents=True, exist_ok=True)

manifests = []

for id, transcription, audio in zip(train_df["id"], train_df["text"], train_audio):
    sf.write(audio_path / f"{id}.wav", audio, 16000)
    manifest_json = {
        "audio_filepath": str(audio_path.name + f"/{id}.wav"),
        "transcription": transcription,
        "language": "en",
        "condition": "real",
        "duration": round(len(audio) / 16000, 3),
        "sample_rate": 16000,
        "utterance_id": str(id),
    }
    manifests.append(manifest_json)


with open(manifest_path, "w") as f:
    for manifest in manifests:
        f.write(json.dumps(manifest) + "\n")

In [17]:
import json
from pathlib import Path

train_path = Path("../data/atc_asr_val")
train_path.mkdir(parents=True, exist_ok=True)
manifest_path = train_path / "manifest.jsonl"
audio_path = train_path / "audio"
audio_path.mkdir(parents=True, exist_ok=True)

manifests = []

for id, transcription, audio in zip(
    validation_df["id"], validation_df["text"], val_audio
):
    sf.write(audio_path / f"{id}.wav", audio, 16000)
    manifest_json = {
        "audio_filepath": str(audio_path.name + f"/{id}.wav"),
        "transcription": transcription,
        "language": "en",
        "condition": "real",
        "duration": round(len(audio) / 16000, 3),
        "sample_rate": 16000,
        "utterance_id": str(id),
    }
    manifests.append(manifest_json)


with open(manifest_path, "w") as f:
    for manifest in manifests:
        f.write(json.dumps(manifest) + "\n")

In [18]:
import json
from pathlib import Path

train_path = Path("../data/atc_asr_test")
train_path.mkdir(parents=True, exist_ok=True)
manifest_path = train_path / "manifest.jsonl"
audio_path = train_path / "audio"
audio_path.mkdir(parents=True, exist_ok=True)

manifests = []

for id, transcription, audio in zip(test_df["id"], test_df["text"], test_audio):
    sf.write(audio_path / f"{id}.wav", audio, 16000)
    manifest_json = {
        "audio_filepath": str(audio_path.name + f"/{id}.wav"),
        "transcription": transcription,
        "language": "en",
        "condition": "real",
        "duration": round(len(audio) / 16000, 3),
        "sample_rate": 16000,
        "utterance_id": str(id),
    }
    manifests.append(manifest_json)


with open(manifest_path, "w") as f:
    for manifest in manifests:
        f.write(json.dumps(manifest) + "\n")

In [19]:
print(
    "Train data length: ",
    round(sum([len(audio) / 16000 for audio in train_audio]) / 60, 3),
)
print(
    "Validation data length: ",
    round(sum([len(audio) / 16000 for audio in val_audio]) / 60, 3),
)
print(
    "Test data length: ",
    round(sum([len(audio) / 16000 for audio in test_audio]) / 60, 3),
)

Train data length:  353.747
Validation data length:  44.859
Test data length:  45.694
